# 🗺️ 31 · Spatial Omics and Multiplexed Imaging
### Not just which cell types are present, but where — and who sits next to whom

**Part J · Research frontier** · Module 31 of 37 (frontier) · builds on 17, 18, 19, 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/31_spatial_omics_and_multiplexed_imaging.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (Colab free tier) |
| **Core frameworks** | NumPy, SciPy, Matplotlib |
| **Prerequisites** | Module 26 (essential), Modules 17, 18/19 |
| **Estimated time** | 80–100 min |
| **Builds on** | Module 26 (colocalization & spatial statistics, extended directly here), Module 17 (dimensionality reduction / clustering ideas) and Modules 18–19 (segmentation, the source of per-cell data in a real pipeline) |
| **Research frontier covered** | Spatial omics & multiplexed imaging — cell typing from multiplexed marker profiles, spatial neighbourhood enrichment, a conceptual graph neural network over spatial graphs, and cell-type niches |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every cell type, marker profile and spatial layout in this notebook is SIMULATED with known ground truth — there is no real spatial-omics data, and nothing is downloaded.

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Explain what multiplexed imaging measures (many protein or RNA channels per tissue section, at
   single-cell resolution) and how it differs from the single- or few-channel imaging the rest of this
   course has focused on.
2. Classify cells into types from their multiplexed per-cell marker intensities, and verify the result
   against known ground truth.
3. Extend Module 26's spatial-statistics toolkit (nearest-neighbour distances, a CSR-style null) into a
   **cell-type neighbourhood-enrichment test**: are two cell types found near each other more, or less, than
   chance would predict?
4. Describe, and implement in miniature, the **message-passing** intuition behind graph neural networks
   operating on a spatial graph of cells.
5. Recognise a spatial **niche** — a region of locally consistent cell-type composition — in a toy dataset,
   and describe why joint imaging + transcriptomic/proteomic integration is where the field is heading next.

## 🗺️ Contents

1. The biological question
2. Environment setup
3. Multiplexed imaging: many markers, one cell
4. Simulating a tissue with known cell types and known spatial structure
5. Cell typing from marker profiles
6. Spatial neighbourhood enrichment: extending Module 26
7. A first look at graph neural networks: one manual message-passing step
8. Niches: regions of consistent cell-type composition
9. From toy to real data
10. Key takeaways, checkpoint, and further practice

## 🧫 The biological question

Module 26 asked whether points of a *single* kind (nuclei, fluorescent spots) are arranged more clumped,
more regular, or more random than chance — a question about the spatial statistics of one population.
Multiplexed spatial imaging adds a second axis entirely: each point (each cell) now carries a **multi-marker
profile** that places it into one of several cell types, and the biological question becomes about the
*joint* arrangement of types in space. Are immune cells systematically found at the edge of a tumour region,
or scattered throughout it? Are two cell types that cooperate functionally also found physically close to
each other more often than chance would predict? Answering these questions needs both a way to assign a cell
type from its marker profile, and a way to test whether the *spatial arrangement* of those types is
meaningful rather than incidental — exactly the two halves this module builds.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 23
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def nearest_neighbor_distances(points: np.ndarray, other: np.ndarray | None = None) -> np.ndarray:
    """For each point, the distance to its nearest neighbour (in ``other`` if given — a CROSS
    nearest-neighbour distance between two populations — else in ``points`` itself)."""
    target = points if other is None else other
    d = np.sqrt(((points[:, None, :] - target[None, :, :]) ** 2).sum(axis=2))
    if other is None:
        np.fill_diagonal(d, np.inf)
    return d.min(axis=1)


def pca(X: np.ndarray, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data. Returns the projected
    scores (N x n_components), the components (n_components x J) and the explained-variance
    ratio of each component."""
    mean = X.mean(axis=0)
    Xc = X - mean
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (U * S)[:, :n_components]
    explained = (S ** 2) / np.sum(S ** 2)
    return scores, Vt[:n_components], explained[:n_components]

---
## 1 · Multiplexed imaging: many markers, one cell

Multiplexed imaging technologies image many more protein (or RNA) markers per
tissue section than conventional fluorescence microscopy's handful of channels — cyclic antibody staining and
imaging (CODEX-style), fluorescence in-situ hybridisation panels that read out dozens to hundreds of RNA
species one molecule at a time (MERFISH-style), or in-situ sequencing-based platforms (10x Genomics Xenium
and similar), all at single-cell (sometimes sub-cellular) spatial resolution. The common output, after
per-cell segmentation (Modules 18–19) and per-cell marker-intensity extraction, is the same kind of table
this module works with directly: one row per cell, one column per marker, plus that cell's spatial
coordinates. This module does not simulate the acquisition physics of any specific platform — it starts from
that per-cell marker table, which is where the spatial-omics analysis proper begins regardless of which
platform produced it.

## 2 · Simulating a tissue with known cell types and known spatial structure

In [ ]:
# ── 2.1 Three cell types, each with a characteristic marker profile ──
# Five SIMULATED markers per cell (think: five antibody or probe channels). Each cell type has its own
# mean marker profile; individual cells vary around it with measurement noise.
MARKER_NAMES = ["marker_1", "marker_2", "marker_3", "marker_4", "marker_5"]
TRUE_CENTROIDS = {
    "T_epithelial": np.array([0.9, 0.1, 0.1, 0.3, 0.2]),
    "T_immune":     np.array([0.1, 0.9, 0.2, 0.1, 0.6]),
    "T_stromal":    np.array([0.2, 0.1, 0.9, 0.6, 0.1]),
}
PROFILE_NOISE = 0.15

def sample_marker_profile(cell_type, rng):
    return np.clip(TRUE_CENTROIDS[cell_type] + rng.normal(0, PROFILE_NOISE, size=5), 0, None)


In [ ]:
# ── 2.2 Two spatial 'niches' with different cell-type compositions ──
# NICHE A (left region of the field): mostly epithelial and immune cells, placed near each other
#   (the true biological signal this module's enrichment test in Section 4 should recover).
# NICHE B (right region): mostly stromal cells, with a little epithelial bleed-through at the boundary.
FIELD_SIZE = 100.0
N_CELLS = 260

def assign_niche_and_type(xy, rng):
    x, _ = xy
    if x < FIELD_SIZE * 0.5:
        return "A", rng.choice(["T_epithelial", "T_immune"], p=[0.55, 0.45])
    else:
        return "B", rng.choice(["T_stromal", "T_epithelial"], p=[0.85, 0.15])

coords = rng.uniform(0, FIELD_SIZE, size=(N_CELLS, 2))
niches, cell_types, profiles = [], [], []
for xy in coords:
    niche, ctype = assign_niche_and_type(xy, rng)
    niches.append(niche)
    cell_types.append(ctype)
    profiles.append(sample_marker_profile(ctype, rng))
niches = np.array(niches)
cell_types = np.array(cell_types)
profiles = np.array(profiles)

fig, ax = plt.subplots(figsize=(5, 5))
for ctype, color in zip(TRUE_CENTROIDS, ["tab:orange", "tab:green", "tab:blue"]):
    sel = cell_types == ctype
    ax.scatter(coords[sel, 0], coords[sel, 1], s=18, label=ctype, color=color, alpha=0.8)
ax.axvline(FIELD_SIZE * 0.5, color="gray", linestyle="--", linewidth=1, label="niche boundary")
ax.set_aspect("equal"); ax.legend(fontsize=8); ax.set_title("Simulated tissue: true cell types in space")
plt.tight_layout(); plt.show()


🔍 **What just happened?** We now have a SIMULATED tissue of 260 cells, each with a known true cell type, a known true niche membership, and a 5-marker profile sampled around that type's characteristic marker signature. A real spatial-omics dataset gives you the coordinates and the marker table; it does NOT give you the true cell-type labels or niche boundaries — those are exactly what Sections 3 and 6 recover, so having them here lets us check our work.

## 3 · Cell typing from marker profiles

With a multiplexed marker table in hand, the first analysis step is almost
always **cell typing**: grouping cells into types from their marker profiles alone, extending the
feature-based classification ideas of Module 15/17 from one or two features to many markers at once. Here we
implement a minimal from-scratch $k$-means clustering — assign each cell to its nearest of $k$ centroids,
recompute centroids as the mean of their assigned cells, repeat — exactly the same alternating-optimisation
structure as the EM thresholding you implemented in Module 18, just in a 5-dimensional feature space instead
of a 1-dimensional intensity histogram.

In [ ]:
# ── 3.1 A minimal from-scratch k-means over marker profiles ──
def cluster_profiles(X, k, n_iter=25, seed=0):
    rng_local = np.random.default_rng(seed)
    centroids = X[rng_local.choice(len(X), size=k, replace=False)].copy()
    assignments = np.zeros(len(X), dtype=int)
    for _ in range(n_iter):
        dists = np.linalg.norm(X[:, None, :] - centroids[None, :, :], axis=2)
        assignments = dists.argmin(axis=1)
        for j in range(k):
            if np.any(assignments == j):
                centroids[j] = X[assignments == j].mean(axis=0)
    return assignments, centroids

cluster_ids, cluster_centroids = cluster_profiles(profiles, k=3, seed=SEED)

# Match cluster indices to true type NAMES by nearest true centroid, purely for readable plotting/scoring —
# the clustering itself never saw a single true label.
type_list = list(TRUE_CENTROIDS)
cluster_to_type = {}
for j in range(3):
    dists_to_true = [np.linalg.norm(cluster_centroids[j] - TRUE_CENTROIDS[t]) for t in type_list]
    cluster_to_type[j] = type_list[int(np.argmin(dists_to_true))]
predicted_types = np.array([cluster_to_type[c] for c in cluster_ids])

purity = np.mean(predicted_types == cell_types)
print(f"cell-typing purity against known ground truth: {purity:.3f}")


🧪 **Ground-truth check.** Because we assigned every cell a known true type
when we simulated it, `purity` is an exact accuracy measurement — the fraction of cells whose unsupervised
cluster assignment, after matching cluster indices to type names by nearest centroid, agrees with the truth.
A real spatial-omics analysis cannot compute this number directly (there is no ground truth), which is
exactly why validating a clustering method on simulated or otherwise-known data, as you just did, matters
before trusting it on real tissue.

✏️ **Your turn.** Increase `PROFILE_NOISE` in Section 2.1 and re-run Sections 2–3. How much per-cell marker
noise can $k$-means tolerate before purity drops substantially? Relate this to why real platforms invest
heavily in reducing per-cell measurement noise (better antibody specificity, more imaging rounds, careful
segmentation) before any downstream spatial analysis is attempted.

## 4 · Spatial neighbourhood enrichment: extending Module 26

Module 26 asked whether a *single* population of points was clustered,
dispersed, or random, using a CSR (complete-spatial-randomness) null built by simulating random point sets of
the same density. Here we ask a two-population question: are cells of type $A$ found closer to cells of type
$B$ than chance would predict, given the actual spatial layout of all the cells?

The **label-permutation null** answers this cleanly: keep every cell's spatial position exactly where it
really is (so the overall point density and spatial layout is preserved exactly), but randomly reshuffle
which cell gets which type label, many times. For each shuffle, recompute the cross-type statistic (here, the
mean nearest-neighbour distance from type $A$ to type $B$, using the toolkit's `nearest_neighbor_distances`
with `other=`). If the *observed*, true-labeled statistic falls far outside the distribution of shuffled
values, the spatial association between $A$ and $B$ is unlikely to be a coincidence of this particular random
layout — it reflects a real relationship between the two types' positions.

In [ ]:
# ── 4.1 Observed cross-type nearest-neighbour distance, for every pair of types ──
def cross_type_stat(coords, type_labels, type_a, type_b):
    pts_a, pts_b = coords[type_labels == type_a], coords[type_labels == type_b]
    if len(pts_a) == 0 or len(pts_b) == 0:
        return np.nan
    return nearest_neighbor_distances(pts_a, other=pts_b).mean()

pairs = [("T_epithelial", "T_immune"), ("T_immune", "T_stromal"), ("T_epithelial", "T_stromal")]
observed = {pair: cross_type_stat(coords, cell_types, *pair) for pair in pairs}
for pair, val in observed.items():
    print(f"{pair[0]:14s} -> {pair[1]:14s} observed mean NN distance: {val:.2f}")


In [ ]:
# ── 4.2 A label-permutation null for each pair, and a two-sided enrichment/avoidance call ──
N_PERM = 500

def permutation_null(coords, type_labels, type_a, type_b, n_perm, seed):
    rng_local = np.random.default_rng(seed)
    null_stats = np.empty(n_perm)
    labels = type_labels.copy()
    for i in range(n_perm):
        shuffled = rng_local.permutation(labels)
        null_stats[i] = cross_type_stat(coords, shuffled, type_a, type_b)
    return null_stats

print(f"{'pair':32s} | observed | null mean ± sd       | call")
for pair in pairs:
    null_stats = permutation_null(coords, cell_types, *pair, n_perm=N_PERM, seed=SEED)
    obs = observed[pair]
    z = (obs - null_stats.mean()) / (null_stats.std() + 1e-9)
    call = "ENRICHED (closer than chance)" if z < -2 else ("AVOIDANT (farther than chance)" if z > 2 else "no clear signal")
    print(f"{pair[0]+' / '+pair[1]:32s} | {obs:8.2f} | {null_stats.mean():6.2f} ± {null_stats.std():5.2f} | {call}")


🔍 **What just happened?** A negative $z$-score means the *observed* nearest-
neighbour distance is *smaller* than the null distribution's typical value — the two types are closer
together than random chance would produce, i.e. spatially **enriched** (co-localized). A positive $z$-score
means they are farther apart than chance — spatially **avoidant**. Given how Section 2.2 built the niches
(epithelial and immune cells sharing niche A; stromal cells concentrated in niche B, with only a little
epithelial overlap), you should see epithelial/immune come out enriched, and immune/stromal come out
avoidant or at least not enriched — matching the spatial structure we built in, by construction.

✏️ **Your turn.** Change Section 2.2's niche-mixing probabilities so that immune and stromal cells are
*deliberately* mixed together in both niches (no true spatial preference). Re-run Section 4. Does the test
correctly report "no clear signal" for that pair now? This is the neighbourhood-enrichment-test analogue of
the negative-control check from Module 30.

### 📄 Research connection — neighbourhood enrichment in practice

This permutation-based neighbourhood test is a simplified version of the **neighbourhood enrichment
analysis** implemented in production spatial-omics toolkits such as **Squidpy** (Palla et al. 2022, *Nature
Methods*), which builds a spatial graph (commonly $k$-nearest-neighbours or Delaunay, exactly the kind of
graph Section 5 uses) and computes a permutation-based enrichment $z$-score for every pair of annotated cell
types across the whole tissue at once — the same logic this module applied to three types, scaled to the
dozens that a real panel typically distinguishes.

## 5 · A first look at graph neural networks: one manual message-passing step

Section 4's enrichment test treats the spatial relationship between types as
a single summary statistic. **Graph neural networks (GNNs)** take the spatial graph more seriously: build a
graph with one node per cell (connected to, say, its $k$ nearest spatial neighbours) and let each node's
feature vector update by aggregating information from its neighbours — **message passing**. One layer of the
simplest possible form:

$$h_i' = \mathrm{AGGREGATE}\big(\{h_j : j \in \mathcal{N}(i)\} \cup \{h_i\}\big)$$

with $\mathrm{AGGREGATE}$ as simple as an unweighted mean. *In words:* each cell's updated representation
becomes a blend of its own marker profile and its spatial neighbours' profiles — which, applied repeatedly
across several layers with *learned* (not just averaged) weights, lets a real GNN learn representations that
depend on a cell's local tissue context, not just its own marker profile in isolation. We do not train a
real GNN here — only one, un-learned, mean-aggregation step, small enough to compute and inspect by hand —
but the geometric idea, and why it denoises noisy per-cell measurements, is exactly the same.

In [ ]:
# ── 5.1 A tiny spatial graph: 12 cells, k-nearest-neighbour edges, one manual message-passing step ──
n_demo = 12
demo_idx = rng.choice(len(coords), size=n_demo, replace=False)
demo_coords = coords[demo_idx]
demo_types = cell_types[demo_idx]
demo_profiles_noisy = profiles[demo_idx]  # each cell's individually noisy observed profile

K_NEIGHBORS = 3
dist_matrix = np.linalg.norm(demo_coords[:, None, :] - demo_coords[None, :, :], axis=2)
np.fill_diagonal(dist_matrix, np.inf)
neighbor_idx = np.argsort(dist_matrix, axis=1)[:, :K_NEIGHBORS]

# ONE manual mean-aggregation message-passing step: each node's new feature = mean of itself + its neighbours.
updated_profiles = np.array([
    np.mean(np.vstack([demo_profiles_noisy[i]] + [demo_profiles_noisy[j] for j in neighbor_idx[i]]), axis=0)
    for i in range(n_demo)
])

# How much closer did each node's feature get to its TRUE type centroid after one message-passing step?
err_before = np.linalg.norm(demo_profiles_noisy - np.array([TRUE_CENTROIDS[t] for t in demo_types]), axis=1)
err_after = np.linalg.norm(updated_profiles - np.array([TRUE_CENTROIDS[t] for t in demo_types]), axis=1)
print(f"{'cell':5s} {'type':14s} {'error before':14s} {'error after':12s}")
for i in range(n_demo):
    print(f"{i:5d} {demo_types[i]:14s} {err_before[i]:14.3f} {err_after[i]:12.3f}")
print(f"\nmean error before message passing: {err_before.mean():.3f}")
print(f"mean error after  message passing: {err_after.mean():.3f}")


🧪 **Ground-truth check.** Because we know each demo cell's true type (and
hence its true marker centroid), we can directly measure whether averaging in spatial neighbours' profiles
moved a noisy observation closer to, or farther from, the truth. When most neighbours share a cell's true
type (the common case inside a niche), mean aggregation acts like a denoising filter — very similar in spirit
to the spatial smoothing filters of Module 11, just operating on a graph instead of a pixel grid. When a
cell's neighbours are a *different* type (a boundary cell between two niches), the same averaging can blur
its representation toward the wrong type — a real and important limitation of naive mean aggregation that
motivates the learned, attention-weighted aggregation real GNNs use instead.

✏️ **Your turn.** Find the demo cells whose error got *worse* after message
passing (if any). Check whether they sit near the niche boundary (`x` close to `FIELD_SIZE * 0.5`) using
`demo_coords`. Does this confirm the boundary-blurring effect described above?

## 6 · Niches: regions of consistent cell-type composition

A spatial **niche** is a region where the local mix of cell types is
relatively consistent — not a single cell type, but a characteristic *composition* (e.g. "mostly epithelial
and immune, roughly half and half"). We can reveal niches directly from the typed, spatially-resolved data by
sliding a window across the tissue and recording the local cell-type composition in each window position —
no model training required, just a spatially-aware histogram.

In [ ]:
# ── 6.1 Sliding-window cell-type composition map ──
WINDOW = 20.0
GRID_N = 12
grid_x = np.linspace(WINDOW / 2, FIELD_SIZE - WINDOW / 2, GRID_N)
grid_y = np.linspace(WINDOW / 2, FIELD_SIZE - WINDOW / 2, GRID_N)

composition_maps = {t: np.zeros((GRID_N, GRID_N)) for t in TRUE_CENTROIDS}
for i, cx in enumerate(grid_x):
    for j, cy in enumerate(grid_y):
        in_window = (np.abs(coords[:, 0] - cx) < WINDOW / 2) & (np.abs(coords[:, 1] - cy) < WINDOW / 2)
        total = in_window.sum()
        if total == 0:
            continue
        for t in TRUE_CENTROIDS:
            composition_maps[t][j, i] = (cell_types[in_window] == t).sum() / total

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, t in zip(axes, TRUE_CENTROIDS):
    im = ax.imshow(composition_maps[t], origin="lower", extent=[0, FIELD_SIZE, 0, FIELD_SIZE],
                   cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"local fraction: {t}", fontsize=10)
    ax.axvline(FIELD_SIZE * 0.5, color="white", linestyle="--", linewidth=1)
fig.colorbar(im, ax=axes, shrink=0.7, label="local fraction of cells")
plt.show()


🔍 **What just happened?** The sliding-window composition maps should recover,
directly from the typed cell coordinates — with no niche label ever given to the analysis — the same
left/right niche boundary we built into the simulation in Section 2.2: epithelial and immune fractions high
on the left, stromal fraction high on the right. This is exactly how real niche-detection methods work: bin
or cluster the *local composition vector* (not the raw cell type) at many spatial locations, then group
locations with similar composition into a small number of recurring niches.

✏️ **Your turn.** Shrink `WINDOW` to 8 and re-run. Does the composition map get noisier (more small-scale
variation) or does the niche boundary stay just as clear? What does this tell you about the trade-off between
spatial resolution and statistical reliability when choosing a niche-detection window size?

## 7 · From toy to real data

To move from this notebook's simulated 5-marker tissue to a real spatial-omics
analysis:

- **Goltsev, Y. et al. (2018). Deep Profiling of Mouse Splenic Architecture with CODEX Multiplexed Imaging.**
  *Cell*, 174(4), 968–981. A foundational multiplexed-antibody imaging platform at single-cell resolution.
- **Chen, K. H., Boettiger, A. N., Moffitt, J. R., Wang, S. & Zhuang, X. (2015). Spatially resolved,
  highly multiplexed RNA profiling in single cells (MERFISH).** *Science*, 348(6233), aaa6090.
- **10x Genomics Xenium** — a current commercial in-situ platform producing exactly the kind of per-cell,
  multi-marker, spatially-resolved table this module worked with directly.
- **Schapiro, D. et al. (2017). histoCAT: analysis of cell phenotypes and interactions in multiplex image
  cytometry data.** *Nature Methods*, 14, 873–876. An early, dedicated tool for exactly the cell-typing and
  neighbourhood-interaction analysis this module built from scratch.
- **Palla, G. et al. (2022). Squidpy: a scalable framework for spatial omics analysis.** *Nature Methods*,
  19, 171–178. The production tool implementing neighbourhood enrichment, spatial graphs, and niche
  detection at the scale of real multiplexed datasets with dozens of markers and tens of thousands of cells.
- Current research increasingly integrates spatial imaging with matched single-cell transcriptomic or
  proteomic data from the same tissue — using the imaging data's spatial precision and the omics data's
  molecular depth together, rather than either alone — and applies graph neural networks (Section 5) with
  learned, rather than simple mean, aggregation to model cell-cell communication across the resulting
  spatial graphs.

## 📌 Key takeaways

- Multiplexed imaging turns every cell into a multi-marker profile with a known spatial position — not just
  "where are the objects" (Module 26) but "what type is each object, and how are the types arranged."
- Cell typing from marker profiles extends Module 15/17's feature-based classification to many markers at
  once, and can be validated by purity against known labels when they exist.
- The label-permutation null extends Module 26's CSR-null logic to a two-population question: are two cell
  types closer together, or farther apart, in space than chance predicts?
- Graph neural networks formalise "blend each cell's profile with its spatial neighbours'" as a learnable
  operation; even unlearned mean aggregation already shows both its denoising benefit and its boundary-
  blurring risk.
- A spatial niche is a region of consistent local cell-type composition, recoverable directly from typed,
  spatially-resolved cells without ever being told the niche boundary in advance.

## ✅ Checkpoint

1. What does a label-permutation null hold fixed, and what does it randomise, when testing for spatial
   enrichment between two cell types?
   <details><summary>Answer</summary>It holds every cell's spatial position fixed exactly as observed, and
   randomises which type label is assigned to which position, breaking any true association between type and
   location while preserving the overall spatial layout and cell counts.</details>

2. In Section 5, under what circumstance did one step of mean-aggregation message passing make a cell's
   feature representation *worse* (farther from its true type centroid) rather than better?
   <details><summary>Answer</summary>When a cell's spatial nearest neighbours are predominantly a different
   cell type than itself — typically a cell near a niche boundary — averaging in those neighbours' profiles
   pulls its representation toward the wrong type.</details>

3. Why can a sliding-window composition map reveal spatial niches without ever being given niche labels?
   <details><summary>Answer</summary>Because it only needs the already-typed cells' positions: computing the
   local fraction of each cell type within a moving window directly surfaces any region where the type mixture
   is locally consistent, which is precisely the definition of a niche — no separate niche-labeling step is
   required.</details>

## 🧠 Questions & discussion

1. The neighbourhood enrichment test in Section 4 treats "closer than chance" as one summary number per cell
   type pair across the whole tissue. What spatial information does this discard that a niche-composition map
   (Section 6) could instead reveal — for instance, could two types be enriched in one region of a tissue and
   avoidant in another?
2. A real GNN's aggregation weights are learned, not fixed at a simple mean. What would you want the network
   to learn to do differently for a boundary cell versus an interior-niche cell, based on what Section 5
   showed about mean aggregation's limitation?
3. Joint imaging + transcriptomics integration combines a technology with excellent spatial resolution but
   limited molecular depth (imaging) with one with deep molecular readout but weaker or no spatial
   information (single-cell RNA sequencing). What kind of analysis becomes possible with both that is not
   possible with either alone?

## 🚀 Bonus — going further

1. Add a fourth cell type that appears in BOTH niches with equal probability, and re-run the enrichment test
   and the niche composition map. Does the sliding-window map still clearly separate the two niches using
   only the three niche-specific types, even with a shared "background" type diluting the signal?
2. Replace the $k$-nearest-neighbour graph in Section 5 with a distance-threshold graph (connect any two
   cells within some fixed radius) and compare how the message-passing result changes for the boundary cells
   you identified in Section 5's "Your turn."
3. Implement a second message-passing layer (apply Section 5's aggregation step twice in a row, using the
   first step's output as the second step's input) and check whether the denoising benefit keeps improving
   or starts to over-smooth, flattening out real local differences between niches — a real and
   well-documented failure mode of stacking too many GNN layers.

## 📝 Lab note

```text
Module 31 — Spatial omics and multiplexed imaging
Date:
Cell-typing purity against known ground truth:
Enrichment/avoidance calls for each type pair:
Mean message-passing error reduction (interior cells vs. boundary cells):
Niche boundary recovered by the sliding-window composition map — did it match the simulated ground truth?
```

## 📚 References

1. Goltsev, Y. et al. (2018). Deep Profiling of Mouse Splenic Architecture with CODEX Multiplexed Imaging.
   *Cell*, 174(4), 968–981.
2. Chen, K. H., Boettiger, A. N., Moffitt, J. R., Wang, S. & Zhuang, X. (2015). Spatially resolved, highly
   multiplexed RNA profiling in single cells. *Science*, 348(6233), aaa6090.
3. Schapiro, D. et al. (2017). histoCAT: analysis of cell phenotypes and interactions in multiplex image
   cytometry data. *Nature Methods*, 14, 873–876.
4. Palla, G. et al. (2022). Squidpy: a scalable framework for spatial omics analysis. *Nature Methods*, 19,
   171–178.
5. Kipf, T. N. & Welling, M. (2017). Semi-Supervised Classification with Graph Convolutional Networks.
   *ICLR*.
6. Hamilton, W. L., Ying, R. & Leskovec, J. (2017). Inductive Representation Learning on Large Graphs
   (GraphSAGE). *NeurIPS*.
7. 10x Genomics (2022). Xenium In Situ platform technical documentation.

## ➡️ Next

Module 32 turns from spatial structure to a question that runs underneath every module in this course: how
much should you trust any single prediction — and how do you decide, efficiently, which images are worth a
human's annotation time?